In [6]:
import pandas as pd
import numpy as np
import os
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import joblib

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 1) LOAD DATA
csv_path = "augmented_ready.csv"   # change if your file is elsewhere
df = pd.read_csv(csv_path)
print("Loaded:", csv_path, "Shape:", df.shape)

# Use CLEAN_TEXT as input and CATEGORY as label
df["text"] = df["CLEAN_TEXT"].astype(str)
df["label"] = df["CATEGORY"].astype(str)

print("\nSample rows:")
print(df[["text", "label"]].head())

print("\nLabel value counts:")
print(df["label"].value_counts())

# 2) ENCODE LABELS
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

print("\nLabel classes mapped to IDs:")
for i, cls in enumerate(le.classes_):
    print(f"{i} → {cls}")

num_labels = len(le.classes_)
os.makedirs("bert_finetuned", exist_ok=True)
joblib.dump(le, "bert_finetuned/label_encoder.pkl")
print("\nSaved label encoder to bert_finetuned/label_encoder.pkl")

# 3) TRAIN / TEST SPLIT
train_df, test_df = train_test_split(
    df[["text", "label_id"]],
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42,
)

print("\nTrain shape:", train_df.shape, "Test shape:", test_df.shape)

# 4) CONVERT TO HF DATASETS
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

# 5) TOKENIZER
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )

train_enc = train_ds.map(tokenize_batch, batched=True)
test_enc  = test_ds.map(tokenize_batch, batched=True)

train_enc = train_enc.rename_column("label_id", "labels")
test_enc  = test_enc.rename_column("label_id", "labels")

train_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 6) MODEL
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

# 7) METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

# 8) TRAINING ARGS  
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=200,
    save_total_limit=2,
    # older versions may not support these, so we drop them:
    # evaluation_strategy, save_strategy, load_best_model_at_end, metric_for_best_model, fp16, report_to
)

# 9) TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics,
)

print("\nStarting training…")
trainer.train()

print("\nEvaluating on test set…")
res = trainer.evaluate(test_enc)
print("Evaluation metrics:", res)

# 10) DETAILED METRICS
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(test_enc)
logits = pred_out.predictions
y_true = pred_out.label_ids
y_pred = np.argmax(logits, axis=1)

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=list(le.classes_),
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion matrix:\n", cm)

# 11) SAVE MODEL + TOKENIZER
trainer.save_model("bert_finetuned")
tokenizer.save_pretrained("bert_finetuned")
print("\nSaved fine-tuned model & tokenizer to ./bert_finetuned")

# 12) OPTIONAL: EXPORT TEST PREDICTIONS
test_texts = test_df["text"].tolist()
test_pred_labels = le.inverse_transform(y_pred)

test_export = pd.DataFrame({
    "text": test_texts,
    "true_label": le.inverse_transform(y_true),
    "pred_label": test_pred_labels,
})
test_export.to_csv("bert_test_predictions.csv", index=False)
print("Exported test predictions to bert_test_predictions.csv")



Torch version: 2.5.1
CUDA available: True
Loaded: augmented_ready.csv Shape: (41068, 30)

Sample rows:
                            text     label
0       fraud bedi krish pvt ltd     Other
1  neft fraud bedi krish pvt ltd  Transfer
2   fraud bedi krish pvt ltd mkt  Shopping
3       fraud bedi krish pvt ltd     Other
4                            nan     Other

Label value counts:
label
Other              30664
Shopping            5681
Transfer            3585
Cash Withdrawal     1138
Name: count, dtype: int64

Label classes mapped to IDs:
0 → Cash Withdrawal
1 → Other
2 → Shopping
3 → Transfer

Saved label encoder to bert_finetuned/label_encoder.pkl

Train shape: (32854, 2) Test shape: (8214, 2)


Map:   0%|          | 0/32854 [00:00<?, ? examples/s]

Map:   0%|          | 0/8214 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting training…


Step,Training Loss
200,0.266700
400,0.004200
600,0.001600
800,0.000900
1000,0.000600
1200,0.000400
1400,0.000400
1600,0.000300
1800,0.000200
2000,0.000200



Evaluating on test set…


Evaluation metrics: {'eval_loss': 7.484776142518967e-05, 'eval_macro_f1': 1.0, 'eval_runtime': 58.8687, 'eval_samples_per_second': 139.531, 'eval_steps_per_second': 1.104, 'epoch': 3.0}

Classification report:
                 precision    recall  f1-score   support

Cash Withdrawal       1.00      1.00      1.00       228
          Other       1.00      1.00      1.00      6133
       Shopping       1.00      1.00      1.00      1136
       Transfer       1.00      1.00      1.00       717

       accuracy                           1.00      8214
      macro avg       1.00      1.00      1.00      8214
   weighted avg       1.00      1.00      1.00      8214


Confusion matrix:
 [[ 228    0    0    0]
 [   0 6133    0    0]
 [   0    0 1136    0]
 [   0    0    0  717]]

Saved fine-tuned model & tokenizer to ./bert_finetuned
Exported test predictions to bert_test_predictions.csv


In [7]:
import os
print(os.listdir("bert_finetuned"))


['config.json', 'label_encoder.pkl', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.txt']


In [8]:
model = DistilBertForSequenceClassification.from_pretrained("bert_finetuned")
tokenizer = DistilBertTokenizerFast.from_pretrained("bert_finetuned")
le = joblib.load("bert_finetuned/label_encoder.pkl")


In [9]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import torch
import numpy as np
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import joblib

# ---------- Load model + tokenizer + label encoder ----------
model = DistilBertForSequenceClassification.from_pretrained("bert_finetuned")
tokenizer = DistilBertTokenizerFast.from_pretrained("bert_finetuned")
le = joblib.load("bert_finetuned/label_encoder.pkl")

BERT_CONF_THRESHOLD = 0.80  # if below this → LLM fallback

app = FastAPI(title="Zenloop Transaction Categorizer")

class Input(BaseModel):
    text: str

def bert_predict(text: str):
    text = str(text).lower().strip()
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).numpy()[0]
    pred_id = int(np.argmax(probs))
    pred_label = le.inverse_transform([pred_id])[0]
    confidence = float(probs[pred_id])
    return pred_label, confidence, probs.tolist()

def fake_llm_reasoner(text: str, bert_label: str, bert_conf: float):
    """
    Placeholder for LLM path.
    Later you'll plug in a real open-source LLM here.
    """
    rationale = f"LLM stub: keeping BERT label '{bert_label}' (conf={bert_conf:.2f}) for text='{text}'"
    # Pretend LLM is a bit less confident for now
    return bert_label, 0.75, rationale

@app.post("/predict")
def predict(inp: Input):
    bert_label, bert_conf, probs = bert_predict(inp.text)

    # High-confidence → trust BERT
    if bert_conf >= BERT_CONF_THRESHOLD:
        return {
            "input": inp.text,
            "final_label": bert_label,
            "final_confidence": bert_conf,
            "source": "bert",
            "bert_confidence": bert_conf,
            "probs": probs,
        }

    # Low-confidence → send to LLM fallback (stub for now)
    llm_label, llm_conf, rationale = fake_llm_reasoner(inp.text, bert_label, bert_conf)
    return {
        "input": inp.text,
        "final_label": llm_label,
        "final_confidence": llm_conf,
        "source": "llm_fallback",
        "bert_label": bert_label,
        "bert_confidence": bert_conf,
        "probs": probs,
        "llm_rationale": rationale,
    }


Writing app.py


In [10]:
!pip install fastapi uvicorn


In [11]:
import os

for root, dirs, files in os.walk(".", topdown=True):
    if "bert_finetuned" in dirs:
        print("Found bert_finetuned at:", os.path.abspath(os.path.join(root, "bert_finetuned")))
    if "app.py" in files:
        print("Found app.py at:", os.path.abspath(os.path.join(root, "app.py")))


Found bert_finetuned at: C:\Users\bhuvi\bert_finetuned
Found app.py at: C:\Users\bhuvi\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\envs\zenloop\Lib\site-packages\jupyterlab_server\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\envs\zenloop\Lib\site-packages\jupyter_console\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\envs\zenloop\Lib\site-packages\jupyter_server_terminals\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\envs\zenloop\Lib\site-packages\notebook\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\envs\zenloop\Lib\site-packages\prompt_toolkit\filters\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\envs\zenloop\Lib\site-packages\pythonwin\pywin\framework\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\Lib\site-packages\flask\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\Lib\site-packages\flask\sansio\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\Lib\site-packages\jupyterlab_server\app.py
Found app.py at: C:\Users\bhuvi\anaconda3\Lib\site-packages\jupyter_con

In [12]:
import requests

tests = [
    "upi flipkart mkt#332 499.00",
    "atm icici chennai 2000.00",
    "fraud_Random Pvt Ltd 3999.99",
]

for t in tests:
    r = requests.post("http://127.0.0.1:8000/predict", json={"text": t})
    print("\nTEXT:", t)
    print(r.json())



TEXT: upi flipkart mkt#332 499.00
{'input': 'upi flipkart mkt#332 499.00', 'final_label': 'Shopping', 'final_confidence': 0.999855637550354, 'source': 'bert', 'bert_confidence': 0.999855637550354, 'probs': [5.2009625505888835e-05, 3.134265716653317e-05, 0.999855637550354, 6.105159991420805e-05]}

TEXT: atm icici chennai 2000.00
{'input': 'atm icici chennai 2000.00', 'final_label': 'Cash Withdrawal', 'final_confidence': 0.9990234375, 'source': 'bert', 'bert_confidence': 0.9990234375, 'probs': [0.9990234375, 0.00015140490722842515, 0.0005509782931767404, 0.0002742286887951195]}

TEXT: fraud_Random Pvt Ltd 3999.99
{'input': 'fraud_Random Pvt Ltd 3999.99', 'final_label': 'Other', 'final_confidence': 0.9208264350891113, 'source': 'bert', 'bert_confidence': 0.9208264350891113, 'probs': [0.0005095354281365871, 0.9208264350891113, 0.07657159864902496, 0.0020924420095980167]}
